# DDPath-KL-IG — Standalone Demo
Diffusion-Schedule Integrated Gradients via KL-IG framework.
Compares three DDPath schedules against the LinearPath baseline across all standard metrics.

In [ ]:
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1 /content/KLIG_V1 2>/dev/null || \
    git -C /content/KLIG_V1 reset --hard origin/claude/general-session-FcgoB
!pip install -e /content/KLIG_V1/infocube-main -q
!pip install captum datasets tqdm scikit-learn -q

In [ ]:
import os, sys, math, json, pickle, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

for _root in ["/content/KLIG_V1/infocube-main", "infocube-main"]:
    if os.path.isdir(_root) and _root not in sys.path:
        sys.path.insert(0, _root)

from klig import KLIntegratedGradients, DDiffusionPath
from klig.core.path import LinearPath
from torchvision.models import resnet50, ResNet50_Weights

In [ ]:
N_IMGS            = 100
N_STEPS           = 50
N_SAMPLES         = 10
N_INSERTION_STEPS = 50
SIGMA_FINAL       = 1 / 256
FORCE_RECOMPUTE   = False
VIS_IMG_IDX       = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CACHE_DIR = Path("/content/drive/MyDrive/ddpath_cache")
except Exception:
    CACHE_DIR = Path("ddpath_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    "KL-IG (adaptive)",
    "DDPath (cosine)",
    "DDPath (linear)",
    "DDPath (quadratic)",
]
COLORS = {
    "KL-IG (adaptive)":    "#333333",
    "DDPath (cosine)":     "#e41a1c",
    "DDPath (linear)":     "#377eb8",
    "DDPath (quadratic)":  "#4daf4a",
}

def _make_path(m):
    if m == "KL-IG (adaptive)":    return LinearPath()
    if m == "DDPath (cosine)":     return DDiffusionPath("cosine")
    if m == "DDPath (linear)":     return DDiffusionPath("linear")
    if m == "DDPath (quadratic)":  return DDiffusionPath("quadratic")
    raise ValueError(m)

In [ ]:
weights    = ResNet50_Weights.IMAGENET1K_V2
model      = resnet50(weights=weights).to(DEVICE).eval()
preprocess = weights.transforms()
imagenet_labels = weights.meta["categories"]
print("ResNet50 loaded")

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denormalize(x): return x.cpu() * _STD + _MEAN

def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0)
    return a.gather(0, idx.unsqueeze(0)).squeeze(0)

from klig.image.stopping import find_sigma_stop

def get_sigma_final(model, x, target):
    return min(max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0/256.0), 1.0)

In [ ]:
_cache_ds = CACHE_DIR / "dataset.pkl"
if not FORCE_RECOMPUTE and _cache_ds.exists():
    with open(_cache_ds, "rb") as f: dataset = pickle.load(f)
    print(f"[cache] dataset  n={len(dataset)}")
else:
    from datasets import load_dataset as _hf
    _ds = _hf("evanarlian/imagenet_1k_resized_256", split="train", streaming=True)
    dataset = []
    for item in tqdm(_ds.take(N_IMGS * 4), desc="loading"):
        img = item["image"]
        if img.mode != "RGB": img = img.convert("RGB")
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)
            tgt  = int(logits.argmax(-1).item())
            conf = logits.softmax(-1)[0, tgt].item()
        if conf > 0.3:
            dataset.append({"x": x, "target": tgt, "idx": len(dataset)})
        if len(dataset) >= N_IMGS: break
    with open(_cache_ds, "wb") as f: pickle.dump(dataset, f)
    print(f"Collected {len(dataset)} images")

In [ ]:
# ── Attribution loop ─────────────────────────────────────────────────────────
_cache_attr = CACHE_DIR / "ddpath_attrs.pkl"

if not FORCE_RECOMPUTE and _cache_attr.exists():
    with open(_cache_attr, "rb") as f: all_attrs = pickle.load(f)
    print("[cache] attrs loaded")
else:
    all_attrs = {m: [] for m in METHODS}
    for row in tqdm(dataset, desc="attributing"):
        x, tgt = row["x"], row["target"]
        x1 = x.squeeze(0).to(DEVICE)
        for m in METHODS:
            # KL-IG (adaptive) uses per-image sigma; DDPath ignores sigma_final
            sigma = get_sigma_final(model, x, tgt) if m == "KL-IG (adaptive)" else SIGMA_FINAL
            ig = KLIntegratedGradients(
                model, path=_make_path(m),
                n_steps=N_STEPS, n_samples=N_SAMPLES,
                sigma_final=sigma, device=DEVICE)
            r = ig.attribute(x1, target=tgt)
            all_attrs[m].append(absmax_collapse(r.attr).cpu())
    with open(_cache_attr, "wb") as f: pickle.dump(all_attrs, f)
    print("Done.")

In [ ]:
# ── 1. Attribution maps (single image) ───────────────────────────────────────
row0   = dataset[VIS_IMG_IDX]
img_np = np.clip(denormalize(row0["x"][0]).permute(1,2,0).numpy(), 0, 1)

fig, axes = plt.subplots(1, 1 + len(METHODS),
                          figsize=(3.2*(1+len(METHODS)), 3.2), facecolor="white")
axes[0].imshow(img_np); axes[0].axis("off")
axes[0].set_title("Original", fontsize=10, fontweight="bold")

for ax, m in zip(axes[1:], METHODS):
    a    = all_attrs[m][VIS_IMG_IDX].numpy()
    vmax = max(float(np.percentile(np.abs(a), 99)), 1e-9)
    ax.imshow(a, cmap="RdBu_r", vmin=-vmax, vmax=vmax); ax.axis("off")
    ax.set_title(m, fontsize=9, fontweight="bold", color=COLORS[m])

plt.suptitle(f"Attribution maps — {imagenet_labels[row0['target']]}", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ── 2. Sparsity (Gini coefficient) ───────────────────────────────────────────
_cache_gini = CACHE_DIR / "ddpath_gini.pkl"

def gini(v):
    v = v.abs().flatten().numpy()
    v = np.sort(v); n = len(v)
    return float((2*np.arange(1,n+1) - n - 1) @ v / (n * v.sum() + 1e-12))

if not FORCE_RECOMPUTE and _cache_gini.exists():
    with open(_cache_gini,"rb") as f: gini_scores = pickle.load(f)
else:
    gini_scores = {m: [gini(all_attrs[m][i]) for i in range(len(dataset))] for m in METHODS}
    with open(_cache_gini,"wb") as f: pickle.dump(gini_scores, f)

fig, ax = plt.subplots(figsize=(7,4), facecolor="white")
for xi, m in enumerate(METHODS):
    v  = gini_scores[m]; mu = np.mean(v)
    ci = 1.96*np.std(v)/len(v)**0.5
    ax.bar(xi, mu, color=COLORS[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
ax.set_xticks(range(len(METHODS))); ax.set_xticklabels(METHODS, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Gini coefficient"); ax.set_title(f"Sparsity (n={len(dataset)})")
plt.tight_layout(); plt.show()

In [ ]:
# ── 3. Insertion / Deletion AUC ──────────────────────────────────────────────
_cache_id = CACHE_DIR / "ddpath_ins_del.pkl"

def insertion_deletion(model, x, attr_map, target, n_steps=N_INSERTION_STEPS):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    order   = attr_map.detach().view(-1).argsort(descending=True)
    pps     = max(1, H*W // n_steps)
    blur    = F.avg_pool2d(x, 31, 1, 15)
    x_ins, x_del = blur.clone(), x.clone()
    ins_s, del_s = [], []
    with torch.no_grad():
        for step in range(n_steps):
            pix = order[step*pps:(step+1)*pps]
            for ch in range(C):
                x_ins[:,ch].reshape(-1)[pix] = x[:,ch].reshape(-1)[pix]
                x_del[:,ch].reshape(-1)[pix] = blur[:,ch].reshape(-1)[pix]
            ins_s.append(model(x_ins).softmax(-1)[0,target].item())
            del_s.append(model(x_del).softmax(-1)[0,target].item())
    return float(np.trapz(ins_s)/n_steps), float(np.trapz(del_s)/n_steps)

if not FORCE_RECOMPUTE and _cache_id.exists():
    with open(_cache_id,"rb") as f: ins_auc, del_auc = pickle.load(f)
else:
    ins_auc = defaultdict(list); del_auc = defaultdict(list)
    for row in tqdm(dataset[:N_IMGS], desc="ins/del"):
        x, tgt = row["x"], row["target"]
        for m in METHODS:
            attr = all_attrs[m][row["idx"]].to(DEVICE).unsqueeze(0)
            i, d = insertion_deletion(model, x, attr, tgt)
            ins_auc[m].append(i); del_auc[m].append(d)
    ins_auc, del_auc = dict(ins_auc), dict(del_auc)
    with open(_cache_id,"wb") as f: pickle.dump((ins_auc, del_auc), f)

fig, axes = plt.subplots(1,2, figsize=(11,4), facecolor="white")
for ax,(title,aucs) in zip(axes,[("Insertion AUC ↑",ins_auc),("Deletion AUC ↓",del_auc)]):
    for xi,m in enumerate(METHODS):
        v=aucs[m]; mu=np.mean(v); ci=1.96*np.std(v)/len(v)**0.5
        ax.bar(xi,mu,color=COLORS[m],alpha=0.85,width=0.6)
        ax.errorbar(xi,mu,yerr=ci,fmt="none",color="black",capsize=4,lw=1.5)
    ax.set_xticks(range(len(METHODS))); ax.set_xticklabels(METHODS,rotation=20,ha="right",fontsize=9)
    ax.set_title(title)
plt.suptitle(f"Insertion / Deletion AUC (n={len(dataset)})", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4. Sensitivity-n ─────────────────────────────────────────────────────────
_cache_sn = CACHE_DIR / "ddpath_sens_n.pkl"

def sensitivity_n(model, x, attr_map, target, n_subsets=50, subset_size=0.1):
    np.random.seed(42)
    attr_flat = attr_map.cpu().detach().view(-1).numpy()
    n_pix     = attr_flat.size
    n_sel     = max(1, int(n_pix * subset_size))
    corrs     = []
    with torch.no_grad():
        f_x = model(x).softmax(-1)[0, target].item()
        for _ in range(n_subsets):
            idx     = np.random.choice(n_pix, n_sel, replace=False)
            x_mask  = x.clone()
            for ch in range(x.shape[1]):
                x_mask[:,ch].reshape(-1)[idx] = 0
            f_mask  = model(x_mask).softmax(-1)[0, target].item()
            delta_f = f_x - f_mask
            delta_a = float(np.abs(attr_flat[idx]).sum())
            corrs.append((delta_f, delta_a))
    df = np.array([c[0] for c in corrs])
    da = np.array([c[1] for c in corrs])
    if df.std() < 1e-9 or da.std() < 1e-9: return 0.0
    return float(np.corrcoef(df, da)[0,1])

if not FORCE_RECOMPUTE and _cache_sn.exists():
    with open(_cache_sn,"rb") as f: sens_n = pickle.load(f)
else:
    sens_n = defaultdict(list)
    for row in tqdm(dataset, desc="sensitivity-n"):
        x, tgt = row["x"], row["target"]
        for m in METHODS:
            attr = all_attrs[m][row["idx"]].to(DEVICE).unsqueeze(0)
            sens_n[m].append(sensitivity_n(model, x, attr, tgt))
    sens_n = dict(sens_n)
    with open(_cache_sn,"wb") as f: pickle.dump(sens_n, f)

fig, ax = plt.subplots(figsize=(7,4), facecolor="white")
for xi,m in enumerate(METHODS):
    v=sens_n[m]; mu=np.mean(v); ci=1.96*np.std(v)/len(v)**0.5
    ax.bar(xi,mu,color=COLORS[m],alpha=0.85,width=0.6)
    ax.errorbar(xi,mu,yerr=ci,fmt="none",color="black",capsize=4,lw=1.5)
ax.set_xticks(range(len(METHODS))); ax.set_xticklabels(METHODS,rotation=20,ha="right",fontsize=9)
ax.set_ylabel("Pearson r"); ax.set_title(f"Sensitivity-n (n={len(dataset)})")
plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Object Focus Ratio (OFR) ──────────────────────────────────────────────
_cache_ofr = CACHE_DIR / "ddpath_ofr.pkl"

def object_focus_ratio(attr_map, mask_pil, img_size=(224,224)):
    import torchvision.transforms.functional as TF
    mask = TF.resize(mask_pil, img_size, interpolation=TF.InterpolationMode.NEAREST)
    mask = torch.tensor(np.array(mask) > 0, dtype=torch.bool)
    a    = attr_map.abs()
    return float(a[mask].sum() / (a.sum() + 1e-9))

if not FORCE_RECOMPUTE and _cache_ofr.exists():
    with open(_cache_ofr,"rb") as f: all_ofr = pickle.load(f)
else:
    from PIL import Image as PILImage
    from datasets import load_dataset as _hf_load
    _seg_ds  = _hf_load("braceletboy/imagenet-s", "ImageNetS919", split="validation", streaming=True)
    _seg_iter = iter(_seg_ds)
    all_ofr  = {m: [] for m in METHODS}
    count = 0
    for row in tqdm(dataset, desc="OFR"):
        try:
            seg = next(_seg_iter)
            mask_pil = seg["annotation"] if isinstance(seg["annotation"], PILImage.Image)                        else PILImage.fromarray(np.array(seg["annotation"]))
            for m in METHODS:
                try:
                    all_ofr[m].append(object_focus_ratio(all_attrs[m][row["idx"]], mask_pil))
                except Exception as e:
                    print(f"  [{m}] img {count}: {e}")
        except StopIteration:
            break
        count += 1
    with open(_cache_ofr,"wb") as f: pickle.dump(all_ofr, f)

fig, ax = plt.subplots(figsize=(7,4), facecolor="white")
for xi,m in enumerate(METHODS):
    v = [x for x in all_ofr[m] if not math.isnan(x)]
    if not v:
        ax.text(xi, 0.02, "no data", ha="center", fontsize=8, color="red"); continue
    mu=np.mean(v); ci=1.96*np.std(v)/len(v)**0.5
    ax.bar(xi,mu,color=COLORS[m],alpha=0.85,width=0.6)
    ax.errorbar(xi,mu,yerr=ci,fmt="none",color="black",capsize=4,lw=1.5)
ax.set_xticks(range(len(METHODS))); ax.set_xticklabels(METHODS,rotation=20,ha="right",fontsize=9)
ax.set_ylabel("OFR"); ax.set_title(f"Object Focus Ratio (n={len(dataset)})")
plt.tight_layout(); plt.show()

In [ ]:
# ── 6. Path-attribution change curves ────────────────────────────────────────
_cache_pc  = CACHE_DIR / "ddpath_curves.pkl"
_cache_pcl = Path("ddpath_curves.pkl")
N_CURVE_IMGS = 20

def _step_signal(m, x1, target, n_steps, n_samples, device, sigma=SIGMA_FINAL):
    path = _make_path(m)
    lv_f = torch.full_like(x1, 2.0 * math.log(sigma))
    ts   = path.steps(n_steps).tolist()
    dt   = 1.0 / n_steps
    sigs = []
    saved = [p.requires_grad for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(False)
    try:
        for t in ts:
            mu_t, lv_t   = path.at(t, x1, lv_f)
            dmu_t, dlv_t = path.derivatives(t, x1, lv_f)
            mu_g = mu_t.detach().requires_grad_(True)
            lv_g = lv_t.detach().requires_grad_(True)
            eps  = torch.randn(n_samples, *x1.shape, device=device)
            xs   = mu_g.unsqueeze(0) + (0.5*lv_g).exp().unsqueeze(0) * eps
            model(xs)[:, target].mean().backward()
            # mu-only signal: avoids logvar cancellation bumps
            sigs.append(float((mu_g.grad * dmu_t * dt).abs().sum()))
    finally:
        for p,s in zip(model.parameters(), saved): p.requires_grad_(s)
    return np.array(sigs)

if not FORCE_RECOMPUTE and (_cache_pc.exists() or _cache_pcl.exists()):
    src = _cache_pc if _cache_pc.exists() else _cache_pcl
    with open(src,"rb") as f: curve_signals = pickle.load(f)
    print(f"[cache] curves from {src}")
else:
    curve_signals = {m: [] for m in METHODS}
    for row in tqdm(dataset[:N_CURVE_IMGS], desc="path-curves"):
        x, tgt = row["x"], row["target"]
        x1 = x.squeeze(0).to(DEVICE)
        for m in METHODS:
            try:
                sigma = get_sigma_final(model, x, tgt) if m == "KL-IG (adaptive)" else SIGMA_FINAL
                curve_signals[m].append(_step_signal(m, x1, tgt, N_STEPS, N_SAMPLES, DEVICE, sigma=sigma))
            except Exception as e:
                print(f"  [{m}]: {e}")
    with open(_cache_pcl,"wb") as f: pickle.dump(curve_signals, f)
    try:
        with open(_cache_pc,"wb") as f: pickle.dump(curve_signals, f)
    except Exception as e:
        print(f"Drive save failed: {e}")

_n  = len(curve_signals[METHODS[0]][0])
xs  = np.linspace(0, 1, _n)
uni = 1.0 / _n

LINESTYLES = {"KL-IG (adaptive)":"-","DDPath (cosine)":"-","DDPath (linear)":"--","DDPath (quadratic)":"-."}  

fig, axes = plt.subplots(1,2,figsize=(13,4),facecolor="white")
for m in METHODS:
    if not curve_signals[m]: continue
    mat   = np.stack(curve_signals[m])
    mean  = mat.mean(0); se = mat.std(0)/math.sqrt(len(mat))
    total = mean.sum()+1e-12; frac = mean/total; fse = se/total
    axes[0].plot(xs,frac,color=COLORS[m],lw=2,ls=LINESTYLES[m],label=m)
    axes[0].fill_between(xs,frac-fse,frac+fse,color=COLORS[m],alpha=0.12)
    axes[1].plot(xs,np.cumsum(frac),color=COLORS[m],lw=2,ls=LINESTYLES[m],label=m)

axes[0].axhline(uni,color="gray",lw=1,ls="--",label="uniform")
axes[0].set_xlabel("Path position s")
axes[0].set_ylabel(r"$|\Delta a(s)|\,/\,\sum_s|\Delta a(s)|$")
axes[0].set_title(f"Attribution change per step  (n={N_CURVE_IMGS})")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25); axes[0].set_xlim(0,1); axes[0].set_ylim(bottom=0)

axes[1].axhline(0.5,color="gray",lw=0.8,ls="--")
axes[1].set_xlabel("Path position s"); axes[1].set_ylabel("Cumulative fraction")
axes[1].set_title("Front-loading comparison")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25); axes[1].set_xlim(0,1)

plt.suptitle("DDPath — path attribution change curves",fontsize=11,fontweight="bold",y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── 7. Cumulative path attribution (interval snapshots) ──────────────────────
T_STEPS = [0.25, 0.5, 0.75, 1.0]
prev_T  = [0.0] + T_STEPS[:-1]

def _cumulative_maps(m, x, target, n_steps, n_samples, device):
    x1   = x.squeeze(0).to(device)
    path = _make_path(m)
    lv_f = torch.full_like(x1, 2.0 * math.log(SIGMA_FINAL))
    ts   = path.steps(n_steps).tolist()
    dt   = 1.0 / n_steps
    snap_at = {f: max(1, round(f*n_steps)) for f in T_STEPS}
    snaps   = {}
    acc_mu  = torch.zeros_like(x1)
    acc_lv  = torch.zeros_like(x1)
    saved   = [p.requires_grad for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(False)
    try:
        for k,t in enumerate(ts):
            mu_t, lv_t   = path.at(t, x1, lv_f)
            dmu, dlv     = path.derivatives(t, x1, lv_f)
            mu_g = mu_t.detach().requires_grad_(True)
            lv_g = lv_t.detach().requires_grad_(True)
            eps  = torch.randn(n_samples, *x1.shape, device=device)
            xs   = mu_g.unsqueeze(0) + (0.5*lv_g).exp().unsqueeze(0)*eps
            model(xs)[:,target].mean().backward()
            acc_mu += mu_g.grad * dmu * dt
            acc_lv += lv_g.grad * dlv * dt
            for f,s in snap_at.items():
                if k+1 == s:
                    snaps[f] = absmax_collapse((acc_mu+acc_lv).clone()).cpu()
    finally:
        for p,s in zip(model.parameters(), saved): p.requires_grad_(s)
    return snaps

print("Computing cumulative maps ...")
row_v  = dataset[VIS_IMG_IDX]
cum_maps = {}
for m in tqdm(METHODS, desc="methods"):
    cum_maps[m] = _cumulative_maps(m, row_v["x"], row_v["target"],
                                    N_STEPS, N_SAMPLES, DEVICE)

img_np = np.clip(denormalize(row_v["x"][0]).permute(1,2,0).numpy(), 0, 1)
N_ROWS = len(METHODS); N_COLS = 1 + len(T_STEPS)
fig, axes = plt.subplots(N_ROWS, N_COLS,
                          figsize=(2.5*N_COLS, 2.5*N_ROWS), facecolor="white")
for mi,m in enumerate(METHODS):
    full  = cum_maps[m][1.0].numpy()
    vmax  = max(float(np.percentile(np.abs(full),99)),1e-9)
    axes[mi,0].imshow(img_np); axes[mi,0].axis("off")
    if mi==0: axes[mi,0].set_title("Original",fontsize=9,fontweight="bold")
    axes[mi,0].set_ylabel(m,fontsize=8,rotation=0,labelpad=90,va="center",fontweight="bold")
    for ci,(tp,t) in enumerate(zip(prev_T,T_STEPS)):
        ax = axes[mi,1+ci]
        dm = cum_maps[m][t].numpy() if tp==0.0 else (cum_maps[m][t]-cum_maps[m][tp]).numpy()
        ax.imshow(dm,cmap="cividis",vmin=-vmax,vmax=vmax); ax.axis("off")
        if mi==0: ax.set_title(f"[{tp:.2f}→{t:.2f}]",fontsize=9,fontweight="bold")
plt.suptitle(f"Incremental path attribution — {imagenet_labels[row_v['target']]}",
             fontsize=11,fontweight="bold")
plt.tight_layout(rect=[0,0,1,0.95]); plt.show()

In [ ]:
# ── 8. Completeness check ────────────────────────────────────────────────────
N_CC = 20; N_PRIOR = 50

def _f_final(x, tgt):
    with torch.no_grad(): return model(x).softmax(-1)[0,tgt].item()

def _f_prior(shape, tgt, n=N_PRIOR):
    with torch.no_grad():
        z = torch.randn(n,*shape,device=DEVICE)
        return model(z).softmax(-1)[:,tgt].mean().item()

rows_cc = []
for row in tqdm(dataset[:N_CC], desc="completeness"):
    x, tgt = row["x"], row["target"]
    x1     = x.squeeze(0).to(DEVICE)
    df     = _f_final(x, tgt) - _f_prior(x1.shape, tgt)
    entry  = {"delta_f": df}
    for m in METHODS:
        sigma = get_sigma_final(model, x, tgt) if m == "KL-IG (adaptive)" else SIGMA_FINAL
        ig = KLIntegratedGradients(model, path=_make_path(m),
                                    n_steps=N_STEPS, n_samples=N_SAMPLES,
                                    sigma_final=sigma, device=DEVICE)
        r  = ig.attribute(x1, target=tgt)
        entry[m] = float(r.attr.sum())
    rows_cc.append(entry)

import pandas as pd
dfs = np.array([r["delta_f"] for r in rows_cc])
print(f"\nCompleteness check  (n={N_CC})")
print(f"{'Method':<28}  {'mean|err|':>9}  {'rel%':>7}  {'pass<1%':>8}")
print("-"*58)
for m in METHODS:
    sums   = np.array([r[m] for r in rows_cc])
    err    = np.abs(sums - dfs)
    rel    = err/(np.abs(dfs)+1e-8)*100
    passed = (rel<1.0).mean()*100
    print(f"{m:<28}  {err.mean():>9.4f}  {rel.mean():>6.2f}%  {passed:>7.1f}%")

In [ ]:
# ── 9. Summary table ─────────────────────────────────────────────────────────
import pandas as pd

ci95 = lambda v: 1.96*np.std(v)/(len(v)**0.5)
rows_sum = []
for m in METHODS:
    row_s = {"Method": m}
    if gini_scores.get(m):
        v=gini_scores[m]; row_s["Gini ↑"]=f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if ins_auc.get(m):
        v=ins_auc[m];  row_s["Ins AUC ↑"]=f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if del_auc.get(m):
        v=del_auc[m];  row_s["Del AUC ↓"]=f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if sens_n.get(m):
        v=sens_n[m];   row_s["Sens-n ↑"]=f"{np.mean(v):.3f}±{ci95(v):.3f}"
    v_ofr=[x for x in all_ofr.get(m,[]) if not math.isnan(x)]
    if v_ofr:
        row_s["OFR ↑"]=f"{np.mean(v_ofr):.3f}±{ci95(v_ofr):.3f}"
    rows_sum.append(row_s)

df_sum = pd.DataFrame(rows_sum).set_index("Method")
print(df_sum.to_string())
df_sum